# Processing Side by Side: Maximum Temperature and Seasonal Climatology

This notebook reproduces the two processing tasks in the source `process.ipynb`. The map projection remains a visualization concern; the numerical fields are calculated independently with xarray and CDO.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from workshop import maximum_temperature, open_tas, seasonal_climatology

DATA = 'data/demo/tas_demo.nc'
tas = open_tas(DATA)
tas

## Task 1: maximum over time

Python: `tas.max('time')`

CDO: `cdo -timmax -subc,273.15 input.nc output.nc`

In [ ]:
python_max = maximum_temperature(tas)
python_max.plot(cmap='coolwarm', vmin=-50, vmax=50)
plt.title('Python/xarray: maximum near-surface air temperature')
plt.show()

In [ ]:
%%bash
set -euo pipefail
mkdir -p outputs/processing/cdo
cdo -L -f nc4c -z zip_4 -timmax -subc,273.15 \
  data/demo/tas_demo.nc outputs/processing/cdo/tas_max_degC.nc

In [ ]:
cdo_max = xr.open_dataset('outputs/processing/cdo/tas_max_degC.nc')['tas'].squeeze()
max_difference = float(np.max(np.abs(cdo_max.values - python_max.values)))
print(f'Maximum-field difference: {max_difference:.3e} degC')
assert max_difference < 2.0e-4

## Task 2: seasonal climatology

Python: `tas.groupby('time.season').mean('time')`

CDO: `cdo -yseasmean -subc,273.15 input.nc output.nc`

In [ ]:
python_seasonal = seasonal_climatology(tas)
python_seasonal.plot(col='season', col_wrap=2, cmap='magma', robust=True)
plt.show()

In [ ]:
%%bash
set -euo pipefail
cdo -L -f nc4c -z zip_4 -yseasmean -subc,273.15 \
  data/demo/tas_demo.nc outputs/processing/cdo/tas_seasonal_degC.nc

In [ ]:
cdo_seasonal = xr.open_dataset('outputs/processing/cdo/tas_seasonal_degC.nc', decode_times=False)['tas'].squeeze()
seasonal_difference = float(np.max(np.abs(cdo_seasonal.values - python_seasonal.values)))
print(f'Seasonal-climatology difference: {seasonal_difference:.3e} degC')
assert seasonal_difference < 2.0e-4
print('PASS: both processing tasks agree within tolerance.')

## Where NCO fits

NCO is particularly effective for selecting variables or hyperslabs and editing metadata:

```bash
ncks -v tas -d time,1572,1931 input.nc tas_1981_2010.nc
ncrename -v tas,tas_anom anomaly.nc
ncatted -a units,tas_anom,o,c,'degC' anomaly.nc
```